# Item ID Embedding — Word2Vec

从 train.csv 的 user 交互序列训练 Word2Vec，生成 43,528 物品的 512-dim 协同 embedding。
对 Word2Vec 未覆盖的物品，用全局均值填充。

In [1]:
import pandas as pd, numpy as np, json, os
from gensim.models import Word2Vec

TRAIN_CSV    = "final/train.csv"
ITEM2ID_PATH = "final/item2id.json"
OUTPUT_DIR   = "final/features"
EMBED_DIM    = 512

os.makedirs(OUTPUT_DIR, exist_ok=True)

### 构造交互序列

In [5]:
train_df = pd.read_csv(TRAIN_CSV)
print(f"交互数: {len(train_df)}, 用户数: {train_df['user_id'].nunique()}, 物品数: {train_df['item_id'].nunique()}")
sequences = []
for user_id, group in train_df.groupby("user_id"):
    item_seq = group.sort_values("timestamp")["item_id"].astype(str).tolist()
    sequences.append(item_seq)
print(f"用户序列数: {len(sequences)}")

交互数: 647950, 用户数: 20030, 物品数: 43518
用户序列数: 20030


In [6]:
# 训练 Word2Vec
model = Word2Vec(
    sentences=sequences,
    vector_size=EMBED_DIM,
    window=7,
    min_count=1,
    sg=1,
    epochs=12,
    workers=4,
    seed=402
)
print(f"Word2Vec 覆盖物品数: {len(model.wv.index_to_key)}")

Word2Vec 覆盖物品数: 43518


In [7]:
# 构建 embedding 矩阵
with open(ITEM2ID_PATH) as f:
    item2id = json.load(f)
total_items = len(item2id)
print(f"总物品数: {total_items}")

collab_matrix = np.zeros((total_items, EMBED_DIM), dtype=np.float32)
covered = 0
for item_str in model.wv.index_to_key:
    idx = int(item_str)
    collab_matrix[idx] = model.wv[item_str]
    covered += 1
print(f"覆盖: {covered}/{total_items}")

总物品数: 43528
覆盖: 43518/43528


In [8]:
# 补全缺失物品
missing_mask = (collab_matrix.sum(axis=1) == 0)
missing_count = missing_mask.sum()
print(f"缺失: {missing_count} (预期 0)")

if missing_count > 0:
    mean_vec = collab_matrix[~missing_mask].mean(axis=0)
    collab_matrix[missing_mask] = mean_vec
    print("已用全局均值填充")

缺失: 10 (预期 0)
已用全局均值填充


In [9]:
np.save(f"{OUTPUT_DIR}/item_id_collab_512.npy", collab_matrix)
print(f"Saved to {OUTPUT_DIR}/item_id_collab_512.npy")
print(f"shape={collab_matrix.shape}, any_zero={(collab_matrix.sum(1)==0).sum()}")

Saved to final/features/item_id_collab_512.npy
shape=(43528, 512), any_zero=0
